"""
Отчёт «Выгрузка ИОР для анализа гипотез за период» (v2 — новая схема БЗ)

Формирует полный Excel-отчёт по ИОР со всеми 67 аналитическими колонками за указанный период.
"""

#### 1. Параметры

In [ ]:
# — Период формирования отчёта ——————————————————————————————————————
incdnt_entry_dt_begin = '2025-01-01'
incdnt_entry_dt_end   = '2025-12-31'

# — Фильтры ——————————————————————————————————————————————————————————
status_filter = None
tb_filter = None
block_filter = None
additional_sql_filter = None

#### 2. Инициализация Spark

In [ ]:
import os
import sys
import datetime
import pandas as pd

from pyspark import SparkContext, SparkConf, HiveContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

_ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# DataLab: /tmp иногда не writable. Spark shuffle/temp файлы — в $HOME.
import os as _os
_SPARK_TMP = _os.path.expanduser('~/.spark-local-dir')
_os.makedirs(_SPARK_TMP, exist_ok=True)
_os.environ['SPARK_LOCAL_DIRS'] = _SPARK_TMP

# DataLab: Spark поднимается локально в контейнере пользователя (local[*]),
# а не submit'ится на YARN. spark.port.maxRetries=100 — на случай коллизий
# портов между несколькими пользователями JupyterHub.
conf = SparkConf().setAppName('ior_hypothesis_v2_' + _ts)
conf.setAll(
    [
        ("spark.ui.enabled",                "true"),
        ("spark.master",                    "local[*]"),
        ("spark.executor.cores",            "2"),
        ("spark.executor.memory",           "8g"),
        ("spark.executor.memoryOverhead",   "1g"),
        ("spark.driver.memory",             "8g"),
        ("spark.driver.maxResultSize",      "8g"),
        ("spark.port.maxRetries",           "100"),
        ("spark.local.dir",                 _SPARK_TMP),
    ]
)
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

# Arrow для zero-copy Spark->Pandas
spark.conf.set('spark.sql.execution.arrow.pyspark.enabled', 'true')
spark.conf.set('spark.sql.execution.arrow.maxRecordsPerBatch', '50000')
print(f'Spark {spark.version} запущен.')

#### 3. Подготовка и фильтрация данных

In [ ]:
print('Загрузка и фильтрация данных...')

DM_NAME = 'arnsdpsbx_t_team_sva_oarb_4.'
MAIN_TABLE = 'd6_base_of_knowledge_ior'

_DT_BEGIN = F.to_timestamp(F.lit(incdnt_entry_dt_begin), 'yyyy-MM-dd')
_DT_END   = F.date_add(F.to_timestamp(F.lit(incdnt_entry_dt_end), 'yyyy-MM-dd'), 1)

report = spark.table(DM_NAME + MAIN_TABLE).filter(
    (F.col('incdnt_entry_dt') >= _DT_BEGIN) &
    (F.col('incdnt_entry_dt') < _DT_END)
)

# Применение динамических фильтров
if status_filter:
    report = report.filter(F.upper(F.col('incdnt_status_name')) == status_filter.upper())
    
if tb_filter:
    report = report.filter(F.lower(F.col('org_struct_lvl_3_name')).contains(tb_filter.lower()))

if block_filter:
    report = report.filter(
        F.lower(F.col('funct_block_lvl_2_name')).contains(block_filter.lower()) |
        F.lower(F.col('funct_block_lvl_3_name')).contains(block_filter.lower()) |
        F.lower(F.col('funct_block_lvl_4_name')).contains(block_filter.lower())
    )

if additional_sql_filter:
    report = report.filter(additional_sql_filter)

report = report.orderBy('incdnt_entry_dt', 'incdnt_sid')

# Вычистка Unicode-мусора
_UNICODE_GARBAGE = r'[\u1128-\uFFFF\x02\x08\x0b]'
report = report.select(
    'incdnt_id', 'incdnt_sid',
    'incdnt_status_name', 'incdnt_autoreg_flag',
    'incdnt_detection_person_name', 'incdnt_source_name',
    'src_type_lvl_1_name', 'src_type_lvl_2_name',
    'incdnt_type_lvl_1_name', 'incdnt_type_lvl_2_name',
    'incdnt_detection_dt', 'incdnt_start_dt', 'incdnt_entry_dt',
    'incdnt_first_validated_dttm', 'incdnt_last_validate_dttm',
    'risk_profile_id', 'risk_profile_name',
    'incdnt_client_type_name', 'incdnt_mistake_cnt',
    'incdnt_appl_num', 'incdnt_agr_num', 'incdnt_agr_sid',
    F.regexp_replace(F.col('incdnt_summary_descr_txt'), _UNICODE_GARBAGE, '').alias('incdnt_summary_descr_txt'),
    F.regexp_replace(F.col('incdnt_full_descr_txt'), _UNICODE_GARBAGE, '').alias('incdnt_full_descr_txt'),
    'org_struct_id',
    'org_struct_lvl_2_name', 'org_struct_lvl_3_name', 'org_struct_lvl_4_name',
    'org_struct_lvl_5_name', 'org_struct_lvl_6_name', 'org_struct_lvl_7_name',
    'org_struct_lvl_8_name', 'org_struct_lvl_9_name', 'org_struct_lvl_10_name',
    'funct_block_id',
    'funct_block_lvl_2_name', 'funct_block_lvl_3_name', 'funct_block_lvl_4_name',
    'process_lvl_1_name', 'process_lvl_2_name', 'process_lvl_3_name', 'process_lvl_4_name',
    'clntpth_lvl_4_name',
    'busn_area_id', 'busn_area_lvl_1_name', 'busn_area_lvl_2_name',
    'incdnt_security_risk_flag', 'incdnt_infrmtn_sys_risk_flag',
    'incdnt_behavior_risk_flag', 'incdnt_model_risk_flag',
    'incdnt_sum',
    'incdnt_drct_dmg_sum', 'incdnt_drct_dmg_cred_rub_amt', 'incdnt_drct_dmg_noncred_rub_amt',
    'incdnt_indrct_dmg_sum', 'incdnt_indrct_dmg_cred_rub_amt', 'incdnt_indrct_dmg_noncred_rub_amt',
    'incdnt_unrlzd_dmg_sum', 'incdnt_unrlzd_dmg_cred_rub_amt', 'incdnt_unrlzd_dmg_noncred_rub_amt',
    'incdnt_thrd_prt_sum', 'incdnt_thrd_prt_cred_rub_amt', 'incdnt_thrd_prt_noncred_rub_amt',
    'incdnt_gain_sum', 'incdnt_gain_cred_rub_amt', 'incdnt_gain_noncred_rub_amt',
    'recovery_rub_amt_aggr'
)
print('Данные успешно подготовлены.')

#### 4. Конвертация и экспорт в Excel

In [ ]:
file_name = f'Выгрузка ИОР для анализа гипотез {incdnt_entry_dt_begin} - {incdnt_entry_dt_end}.xlsx'
df = report.toPandas()
df.columns = [c.lower() for c in df.columns]

RENAME = {
    'incdnt_id': 'Идентификационный ключ инцидента операционного риска',
    'incdnt_sid': 'Идентификатор события',
    'incdnt_status_name': 'Статус события',
    'incdnt_autoreg_flag': 'Признак авторегистрации инцидента',
    'incdnt_detection_person_name': 'Кем выявлено событие',
    'incdnt_source_name': 'Название источника',
    'src_type_lvl_1_name': 'Тип источника инцидента (уровень 1)',
    'src_type_lvl_2_name': 'Тип источника инцидента (уровень 2)',
    'incdnt_type_lvl_1_name': 'Тип события – уровень 1',
    'incdnt_type_lvl_2_name': 'Тип события – уровень 2',
    'incdnt_detection_dt': 'Дата обнаружения (Событие)',
    'incdnt_start_dt': 'Дата начала инцидента операционного риска',
    'incdnt_entry_dt': 'Дата ввода (Событие)',
    'incdnt_first_validated_dttm': 'Дата первого подтверждения',
    'incdnt_last_validate_dttm': 'Дата последнего подтверждения',
    'risk_profile_id': 'Идентификатор профиля риска',
    'risk_profile_name': 'Наименование профиля риска',
    'incdnt_client_type_name': 'Тип клиента',
    'incdnt_mistake_cnt': 'Количество ошибок',
    'incdnt_appl_num': 'Номер заявки',
    'incdnt_agr_num': 'Номер договора',
    'incdnt_agr_sid': 'Идентификатор договора',
    'incdnt_summary_descr_txt': 'Предварительное описание',
    'incdnt_full_descr_txt': 'Подробное описание',
    'org_struct_id': 'Идентификатор оргструктуры',
    'org_struct_lvl_2_name': 'Орг. структура – уровень 2 (Терр. структура / Департамент ДЗО)',
    'org_struct_lvl_3_name': 'Орг. структура – уровень 3 (Блок / ТБ / ПЦП)',
    'org_struct_lvl_4_name': 'Орг. структура – уровень 4 (Дивизион / Департамент)',
    'org_struct_lvl_5_name': 'Орг. структура – уровень 5',
    'org_struct_lvl_6_name': 'Орг. структура – уровень 6',
    'org_struct_lvl_7_name': 'Орг. структура – уровень 7',
    'org_struct_lvl_8_name': 'Орг. структура – уровень 8',
    'org_struct_lvl_9_name': 'Орг. структура – уровень 9',
    'org_struct_lvl_10_name': 'Орг. структура – уровень 10',
    'funct_block_id': 'Идентификатор функционального блока',
    'funct_block_lvl_2_name': 'Функциональный блок – уровень 2',
    'funct_block_lvl_3_name': 'Функциональный блок – уровень 3',
    'funct_block_lvl_4_name': 'Функциональный блок – уровень 4',
    'process_lvl_1_name': 'Процесс – уровень 1',
    'process_lvl_2_name': 'Процесс – уровень 2',
    'process_lvl_3_name': 'Процесс – уровень 3',
    'process_lvl_4_name': 'Процесс – уровень 4 (Наименование процесса)',
    'clntpth_lvl_4_name': 'Клиентский путь – уровень 4',
    'busn_area_id': 'Идентификационный ключ направления деятельности',
    'busn_area_lvl_1_name': 'Направление деятельности банка',
    'busn_area_lvl_2_name': 'Поднаправление деятельности банка',
    'incdnt_security_risk_flag': 'Связь с ИБ-риском',
    'incdnt_infrmtn_sys_risk_flag': 'Связь с риском информационных систем',
    'incdnt_behavior_risk_flag': 'Связь с поведенческим риском',
    'incdnt_model_risk_flag': 'Связь с модельным риском',
    'incdnt_sum': 'Общая сумма всех последствий (руб.)',
    'incdnt_drct_dmg_sum': 'Прямая потеря – итого (руб.)',
    'incdnt_drct_dmg_cred_rub_amt': 'Прямая потеря – с кредитным риском (руб.)',
    'incdnt_drct_dmg_noncred_rub_amt': 'Прямая потеря – без кредитного риска (руб.)',
    'incdnt_indrct_dmg_sum': 'Косвенная потеря – итого (руб.)',
    'incdnt_indrct_dmg_cred_rub_amt': 'Косвенная потеря – с кредитным риском (руб.)',
    'incdnt_indrct_dmg_noncred_rub_amt': 'Косвенная потеря – без кредитного риска (руб.)',
    'incdnt_unrlzd_dmg_sum': 'Нереализовавшаяся потеря – итого (руб.)',
    'incdnt_unrlzd_dmg_cred_rub_amt': 'Нереализовавшаяся потеря – с кредитным риском (руб.)',
    'incdnt_unrlzd_dmg_noncred_rub_amt': 'Нереализовавшаяся потеря – без кредитного риска (руб.)',
    'incdnt_thrd_prt_sum': 'Потеря третьих лиц – итого (руб.)',
    'incdnt_thrd_prt_cred_rub_amt': 'Потеря третьих лиц – с кредитным риском (руб.)',
    'incdnt_thrd_prt_noncred_rub_amt': 'Потеря третьих лиц – без кредитного риска (руб.)',
    'incdnt_gain_sum': 'Прибыль – итого (руб.)',
    'incdnt_gain_cred_rub_amt': 'Прибыль – с кредитным риском (руб.)',
    'incdnt_gain_noncred_rub_amt': 'Прибыль – без кредитного риска (руб.)',
    'recovery_rub_amt_aggr': 'Возмещение – итого по инциденту (руб.)'
}

df = df.rename(columns=RENAME)
print(f'Готовлю {len(df):,} строк к экспорту в Excel...')
df.to_excel(file_name, sheet_name='Отчет_ОпРиски', index=False, engine='openpyxl')
print(f'Отчёт успешно сохранен: {file_name} ({len(df):,} строк)')
spark.stop()